<a href="https://colab.research.google.com/github/alirezzasarkar/analyze_cryptocurrency/blob/main/fundamental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install newsapi-python transformers pandas

In [ ]:
from newsapi import NewsApiClient
from transformers import pipeline
import pandas as pd
from datetime import datetime, timedelta

# Initialize NewsAPI Client (It is recommended to use environment variables for the API key)
newsapi = NewsApiClient(api_key='3e3849190f8f4e91ab89e13cc5bd0c8c')

# Initialize sentiment analysis model using Hugging Face Transformers
sentiment = pipeline('sentiment-analysis')

# Define a dictionary of cryptocurrency tickers
crypto_tickers = {
    1: 'BTC',
    2: 'ETH',
    3: 'ADA',
    4: 'XRP',
    5: 'SOL'
}

# Display available cryptocurrencies
print("List of available cryptocurrencies:")
for key, value in crypto_tickers.items():
    print(f"{key}: {value}")

# Get user input for selecting a cryptocurrency
try:
    selected_crypto_index = int(input("Enter the number of the desired cryptocurrency: "))
    selected_crypto = crypto_tickers.get(selected_crypto_index)
    if selected_crypto is None:
        raise ValueError
except ValueError:
    print("Invalid selection! Please enter a valid number.")
    exit()

# Let the user choose the type of analysis
print("\nSelect the type of analysis:")
print("1: Short-term (hourly data) - Last 3 days")
print("2: Medium-term (daily data) - Last 7 days")
print("3: Long-term (daily data) - Last 30 days")

# Get user input for the analysis type
try:
    selected_analysis = int(input("Enter the analysis type number: "))
except ValueError:
    print("Invalid analysis selection!")
    exit()

# Define static start dates based on the selected analysis type
if selected_analysis == 1:  # Short-term
    start_date = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')
elif selected_analysis == 2:  # Medium-term
    start_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
elif selected_analysis == 3:  # Long-term
    start_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
else:
    print("Invalid analysis selection. Please try again.")
    exit()

# Function to fetch cryptocurrency news using NewsAPI with pagination support
def fetch_crypto_news(ticker, start_date, page_size=10, max_pages=5):
    all_articles = []
    for page in range(1, max_pages + 1):
        response = newsapi.get_everything(
            q=ticker,
            language='en',
            from_param=start_date,
            sort_by='relevancy',
            page_size=page_size,
            page=page
        )
        articles = response.get('articles', [])
        if not articles:
            break  # Stop if no articles are retrieved
        all_articles.extend(articles)
        if len(articles) < page_size:
            break
    return all_articles

# Function to process news articles (extracts title, description, source, and published date)
def process_articles(articles):
    processed_articles = []
    for article in articles:
        description = article.get('description', "")
        title = article.get('title', "")
        source = article.get('source', {}).get('name', "Unknown")  # Extract news source
        published_at = article.get('publishedAt', "")
        processed_articles.append({
            'title': title,
            'description': description,
            'source': source,
            'published_at': published_at,
            'url': article.get('url', "")
        })
    return processed_articles

# Function to determine the overall sentiment of a news article
def get_overall_sentiment(article, threshold=0.6):
    title_sent = article.get('sentiment_title')
    title_conf = article.get('confidence_title', 0)
    desc_sent = article.get('sentiment_description')
    desc_conf = article.get('confidence_description', 0)

    if title_sent == desc_sent:
        return title_sent
    else:
        # Choose the sentiment with the higher confidence score
        if title_conf >= desc_conf:
            chosen = title_sent
            chosen_conf = title_conf
        else:
            chosen = desc_sent
            chosen_conf = desc_conf
        if chosen_conf < threshold:
            return 'Neutral'
        return chosen

# Function to assign weights to news articles based on source credibility and publication time
def calculate_weight(article):
    # Define weight values for key crypto news sources
    source_weights = {
         'CoinDesk': 1.5,
         'CoinTelegraph': 1.5,
         'CryptoSlate': 1.3,
         'Decrypt': 1.3,
         'Bitcoin Magazine': 1.4,
         'Bloomberg': 1.4,
         'Reuters': 1.4,
         'CNN': 1.2,
         'Unknown': 1.0
    }
    source = article.get('source', 'Unknown')
    source_weight = source_weights.get(source, 1.0)

    # Apply time-based weighting (recent news should have more impact)
    try:
        # Convert publication time (ISO format) to datetime
        published_time = datetime.strptime(article['published_at'], '%Y-%m-%dT%H:%M:%SZ')
    except Exception:
        published_time = datetime.now()

    # Calculate time difference in hours
    time_diff_hours = (datetime.now() - published_time).total_seconds() / 3600
    time_weight = 0.98 ** time_diff_hours  # Exponential decay in weight over time

    return source_weight * time_weight

# Function to analyze news sentiment and calculate weighted sentiment scores
def analyze_sentiment_weighted(articles):
    weighted_score = 0.0
    for article in articles:
        # Perform sentiment analysis on title
        sentiment_title = sentiment(article['title'])[0]

        # Perform sentiment analysis on description (if available)
        if article['description']:
            sentiment_description = sentiment(article['description'])[0]
        else:
            sentiment_description = {'label': 'Neutral', 'score': 0.0}

        # Determine the overall sentiment using title and description analysis
        overall_sent = get_overall_sentiment({
            'sentiment_title': sentiment_title['label'],
            'confidence_title': sentiment_title['score'],
            'sentiment_description': sentiment_description['label'],
            'confidence_description': sentiment_description['score']
        })

        # Compute article weight
        weight = calculate_weight(article)

        # Assign a score based on sentiment: Positive (+1), Negative (-1), Neutral (0)
        if overall_sent == 'POSITIVE':
            score = 1
        elif overall_sent == 'NEGATIVE':
            score = -1
        else:
            score = 0

        # Compute weighted sentiment score
        weighted_score += score * weight

        # Store additional information in the article dictionary
        article['overall_sentiment'] = overall_sent
        article['weight'] = weight
        article['weighted_score'] = score * weight

    return weighted_score, articles

# Fetch cryptocurrency news articles
print(f"\nFetching news related to {selected_crypto} from {start_date}...")
articles = fetch_crypto_news(selected_crypto, start_date, page_size=10, max_pages=5)

# Process the fetched articles
processed_articles = process_articles(articles)

# Perform sentiment analysis and calculate weighted scores
total_weighted_score, analyzed_articles = analyze_sentiment_weighted(processed_articles)

# Display the total weighted sentiment score
print(f"\nTotal Weighted Sentiment Score: {total_weighted_score}")

# Determine final recommendation based on sentiment score
if total_weighted_score > 0.5:
    recommendation = "Buy Recommendation"
elif total_weighted_score < -0.5:
    recommendation = "Sell Recommendation"
else:
    recommendation = "No Recommendation"

print(f"\nAnalysis Result: {recommendation}")

# Save analysis results to a CSV file
df = pd.DataFrame(analyzed_articles, columns=[
    'title', 'description', 'source', 'published_at', 'url',
    'sentiment_title', 'confidence_title', 'sentiment_description', 'confidence_description',
    'overall_sentiment', 'weight', 'weighted_score'
])

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


List of available cryptocurrencies:
1: BTC
2: ETH
3: ADA
4: XRP
5: SOL
Enter the number of the desired cryptocurrency: 1

Select the type of analysis:
1: Short-term (hourly data) - Last 3 days
2: Medium-term (daily data) - Last 7 days
3: Long-term (daily data) - Last 30 days
Enter the analysis type number: 1

Fetching news related to BTC from 2025-02-18...

Total Weighted Sentiment Score: -12.42381024354896

Analysis Result: Sell Recommendation
